In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

First for sentiment classification

In [ ]:
def generate_label(system_prompt, user_prompt, max_new_tokens=20):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]


    tokenized_chat = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)


    input_ids = tokenized_chat['input_ids']
    attention_mask = tokenized_chat['attention_mask']

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )


    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return response

In [ ]:
sentiment_system_prompt = """
You are a text classification assistant.
Read the review and output exactly one label from:
positive, neutral, negative

Do not explain.
Do not add punctuation.
Output only the label.
""".strip()

In [ ]:
import pandas as pd

sent_df = pd.read_csv("/content/sentiment_real_reviews_1000.csv")
sent_df = sent_df[["reviews", "sentiment_label"]].dropna()

print(sent_df.head())
print(sent_df["sentiment_label"].value_counts())

In [ ]:
sample_reviews = [
    "This product is excellent and works perfectly. I am very happy with it.",
    "The item is okay. Not bad, but nothing special.",
    "Very disappointing product. It stopped working after two days."
]

for review in sample_reviews:
    pred = generate_label(
        sentiment_system_prompt,
        f"Review: {review}"
    )
    print("Review:", review)
    print("Predicted sentiment:", pred)
    print("-" * 60)

In [ ]:
sentiment_predictions = []

for review in sent_df["reviews"]:
    pred = generate_label(
        sentiment_system_prompt,
        f"Review: {review}"
    ).lower().strip()
    sentiment_predictions.append(pred)

sent_df["predicted_sentiment"] = sentiment_predictions

In [ ]:
valid_sentiment_labels = {"positive", "neutral", "negative"}

def clean_sentiment_label(x):
    x = str(x).lower().strip()
    if "positive" in x:
        return "positive"
    elif "neutral" in x:
        return "neutral"
    elif "negative" in x:
        return "negative"
    return "unknown"

sent_df["predicted_sentiment"] = sent_df["predicted_sentiment"].apply(clean_sentiment_label)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(sent_df["sentiment_label"], sent_df["predicted_sentiment"]))
print("\nClassification Report:\n")
print(classification_report(sent_df["sentiment_label"], sent_df["predicted_sentiment"]))
print("\nConfusion Matrix:\n")
print(confusion_matrix(sent_df["sentiment_label"], sent_df["predicted_sentiment"]))

Now Checking on sarcasm dataset

In [ ]:
sarcasm_system_prompt = """
You are a sarcasm detection assistant.
Read the product review and output exactly one label from:
sarcastic, standard

'sarcastic' means the review uses irony, contradiction, or implied meaning.
'standard' means the review is literal and non-sarcastic.

Do not explain.
Do not add punctuation.
Output only the label.
""".strip()

In [ ]:
sar_df = pd.read_csv("/content/sarcasm_mixed_reviews_1000.csv")
sar_df = sar_df[["reviews", "style"]].dropna()

print(sar_df.head())
print(sar_df["style"].value_counts())

In [ ]:
sample_reviews = [
    "This product is fantastic and works perfectly.",
    "Wow, amazing charger. It stopped working in one day.",
    "Absolutely brilliant. I love how it stopped working immediately.",
    "The product is average, not too bad and not too good.",
    "disappointing charger. I thought it would stop working in one day, but is working absolutely fine."
]

for review in sample_reviews:
    pred = generate_label(
        sarcasm_system_prompt,
        f"Review: {review}"
    )
    print("Review:", review)
    print("Predicted style:", pred)
    print("-" * 60)

In [ ]:
sarcasm_predictions = []

for review in sar_df["reviews"]:
    pred = generate_label(
        sarcasm_system_prompt,
        f"Review: {review}"
    ).lower().strip()
    sarcasm_predictions.append(pred)

sar_df["predicted_style"] = sarcasm_predictions

In [ ]:
valid_sarcasm_labels = {"sarcastic", "standard"}

def clean_sarcasm_label(x):
    x = str(x).lower().strip()
    if "sarcastic" in x:
        return "sarcastic"
    elif "standard" in x:
        return "standard"
    return "unknown"

sar_df["predicted_style"] = sar_df["predicted_style"].apply(clean_sarcasm_label)

In [ ]:
print("Accuracy:", accuracy_score(sar_df["style"], sar_df["predicted_style"]))
print("\nClassification Report:\n")
print(classification_report(sar_df["style"], sar_df["predicted_style"]))
print("\nConfusion Matrix:\n")
print(confusion_matrix(sar_df["style"], sar_df["predicted_style"]))

In [ ]:
sent_df.to_csv("/content/llama_sentiment_predictions.csv", index=False)
sar_df.to_csv("/content/llama_sarcasm_predictions.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


models = ["SVM", "Fine-tuned\nBERT", "Zero-shot\nBERT", "LLaMA-3"]

sentiment_accuracy = [89, 92, 50, 49]
sarcasm_accuracy = [99, 99, 61, 70]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(9, 5.5))

bars1 = plt.bar(
    x - width/2,
    sentiment_accuracy,
    width,
    label="Sentiment Accuracy"
)

bars2 = plt.bar(
    x + width/2,
    sarcasm_accuracy,
    width,
    label="Sarcasm Accuracy"
)

plt.xlabel("Models")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Comparison of Sentiment and Sarcasm Models")
plt.xticks(x, models)
plt.yticks(np.arange(0, 101, 10))
plt.ylim(0, 105)
plt.legend()


for bar in list(bars1) + list(bars2):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 1,
        f"{height}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.show()